# Project 43 — Explainable Disaster Severity Assessment
## Notebook 02: Model Training
**C-DAC Mohali | ML to Generative AI & LLM**  
Team: Anuksha | Rishika | Ipshita  
Dataset: AIDERv2 | Backbone: EfficientNet B0 / B1 / B3

## 1. Imports & reproducibility

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0, EfficientNetB1, EfficientNetB3
from sklearn.metrics import classification_report, confusion_matrix

# ── Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── GPU memory growth (prevents crashes on local machines)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU(s) found: {[g.name for g in gpus]}")
else:
    print("No GPU found — running on CPU. B0 only recommended locally.")

print("Python  :", sys.version)
print("TF      :", tf.__version__)


## 2. Configuration
All paths and hyperparameters in one place. Edit here only.

In [ ]:
# ── Paths (relative — always run Jupyter from project root)
TRAIN_DIR  = os.path.join("data", "Train")
VAL_DIR    = os.path.join("data", "Val")
TEST_DIR   = os.path.join("data", "Test")
MODELS_DIR = "models"
OUT_DIR    = "outputs"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUT_DIR,    exist_ok=True)

# ── Image & training config
IMG_SIZE_B0B1 = (224, 224)
IMG_SIZE_B3   = (300, 300)
BATCH_SIZE    = 32
CLASSES       = ['Earthquake', 'Fire', 'Flood', 'Normal']
NUM_CLASSES   = len(CLASSES)

# ── Phase 1 hyperparameters
P1_EPOCHS = 20
P1_LR     = 1e-3
DROPOUT   = 0.3

# ── Phase 2 hyperparameters
P2_EPOCHS        = 15
P2_LR            = 1e-5
P2_UNFREEZE_LAST = 20

print("Train :", TRAIN_DIR)
print("Val   :", VAL_DIR)
print("Test  :", TEST_DIR)


## 3. Dataset inspection

In [ ]:
def count_images(base_dir, classes):
    counts = {}
    for cls in classes:
        path = os.path.join(base_dir, cls)
        n = len([f for f in os.listdir(path)
                 if f.lower().endswith(('.jpg','.jpeg','.png'))]) if os.path.exists(path) else 0
        counts[cls] = n
    return counts

splits = {'Train': TRAIN_DIR, 'Val': VAL_DIR, 'Test': TEST_DIR}
split_counts = {s: count_images(d, CLASSES) for s, d in splits.items()}

df_counts = pd.DataFrame(split_counts).T
print(df_counts)
print(f"\nTotal training images: {df_counts.loc['Train'].sum()}")

# ── Bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#E24B4A', '#EF9F27', '#378ADD', '#1D9E75']

for i, (split, counts) in enumerate(split_counts.items()):
    axes[i].bar(counts.keys(), counts.values(), color=colors)
    axes[i].set_title(f'{split} set', fontsize=12)
    axes[i].set_ylabel('Image count')
    axes[i].tick_params(axis='x', rotation=15)
    for j, (cls, v) in enumerate(counts.items()):
        axes[i].text(j, v + 20, str(v), ha='center', fontsize=9)

plt.suptitle('AIDERv2 class distribution', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()


## 4. Class weights
Handle class imbalance — Earthquake is underrepresented.

In [ ]:
train_counts = count_images(TRAIN_DIR, CLASSES)
total = sum(train_counts.values())
class_weight = {i: total / (NUM_CLASSES * train_counts[cls])
                for i, cls in enumerate(CLASSES)}

print("Class weights:")
for i, cls in enumerate(CLASSES):
    print(f"  {i} {cls:12s}: {class_weight[i]:.4f}")


## 5. tf.data pipeline
Augmentation on training set only. Val/Test get normalization only.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(directory, img_size, shuffle=False, augment=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        directory,
        image_size=img_size,
        batch_size=BATCH_SIZE,
        label_mode='categorical',
        shuffle=shuffle,
        seed=SEED,
        class_names=CLASSES
    )

    # Normalize to [0, 1]
    def normalize(img, lbl):
        return tf.cast(img, tf.float32) / 255.0, lbl

    ds = ds.map(normalize, num_parallel_calls=AUTOTUNE)

    if augment:
        aug = keras.Sequential([
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1),
            layers.RandomBrightness(0.1),
        ], name="augmentation")
        def apply_aug(img, lbl):
            return aug(img, training=True), lbl
        ds = ds.map(apply_aug, num_parallel_calls=AUTOTUNE)

    return ds.prefetch(AUTOTUNE)

train_ds_b0 = make_dataset(TRAIN_DIR, IMG_SIZE_B0B1, shuffle=True, augment=True)
val_ds_b0   = make_dataset(VAL_DIR,   IMG_SIZE_B0B1, shuffle=False, augment=False)
test_ds_b0  = make_dataset(TEST_DIR,  IMG_SIZE_B0B1, shuffle=False, augment=False)

print("Classes :", train_ds_b0.class_names)
print("Train batches:", len(train_ds_b0))
print("Val batches:  ", len(val_ds_b0))
print("Test batches: ", len(test_ds_b0))


## 6. Sample images

In [ ]:
import glob

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, cls in enumerate(CLASSES):
    # try jpg then png
    files = (glob.glob(os.path.join(TRAIN_DIR, cls, '*.jpg')) +
             glob.glob(os.path.join(TRAIN_DIR, cls, '*.jpeg')) +
             glob.glob(os.path.join(TRAIN_DIR, cls, '*.png')))
    if files:
        from PIL import Image as PILImage
        img = PILImage.open(files[0])
        axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=12)
    axes[i].axis('off')

plt.suptitle('Sample training images — one per class', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'sample_images.png'), dpi=150, bbox_inches='tight')
plt.show()


## 7. Model builder & callbacks

In [ ]:
def build_model(backbone_fn, img_size, freeze_base=True, name='model'):
    """Build EfficientNet transfer learning model."""
    inputs = keras.Input(shape=(*img_size, 3))

    base = backbone_fn(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs
    )
    base.trainable = not freeze_base

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name=name)
    return model, base


def make_callbacks(model_name, patience_es=5, patience_lr=3):
    """Standard callback set for both training phases."""
    ckpt_path = os.path.join(MODELS_DIR, f'{model_name}.h5')
    return [
        keras.callbacks.ModelCheckpoint(
            ckpt_path,
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=patience_es,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=patience_lr,
            min_lr=1e-7,
            verbose=1
        ),
        keras.callbacks.CSVLogger(
            os.path.join(OUT_DIR, f'{model_name}_log.csv')
        ),
    ]

print("Model builder and callbacks ready.")


## 8. Plot helpers

In [ ]:
def plot_history(h1, h2, model_name):
    """Plot Phase 1 + Phase 2 training curves together."""
    acc  = h1.history['accuracy']  + h2.history['accuracy']
    val  = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss = h1.history['loss'] + h2.history['loss']
    vloss= h1.history['val_loss'] + h2.history['val_loss']
    ep   = range(1, len(acc)+1)
    p2_start = len(h1.history['accuracy'])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(ep, acc,  label='Train acc')
    ax1.plot(ep, val,  label='Val acc')
    ax1.axvline(p2_start, color='gray', linestyle='--', alpha=0.6, label='Phase 2 start')
    ax1.set_title(f'{model_name} — Accuracy')
    ax1.set_xlabel('Epoch'); ax1.legend()

    ax2.plot(ep, loss,  label='Train loss')
    ax2.plot(ep, vloss, label='Val loss')
    ax2.axvline(p2_start, color='gray', linestyle='--', alpha=0.6, label='Phase 2 start')
    ax2.set_title(f'{model_name} — Loss')
    ax2.set_xlabel('Epoch'); ax2.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f'{model_name}_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {model_name}_curves.png")


def evaluate_model(model, test_ds, model_name):
    """Confusion matrix + classification report + saves outputs."""
    y_true, y_pred = [], []
    for imgs, lbls in test_ds:
        preds = model.predict(imgs, verbose=0)
        y_true.extend(np.argmax(lbls.numpy(), axis=1))
        y_pred.extend(np.argmax(preds, axis=1))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(f'{model_name} — Confusion matrix')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f'{model_name}_confusion.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # Classification report
    report = classification_report(y_true, y_pred, target_names=CLASSES)
    print(report)
    with open(os.path.join(OUT_DIR, f'{model_name}_report.txt'), 'w') as f:
        f.write(report)

    return report

print("Plot helpers ready.")


## 9. EfficientNetB0 — Phase 1 (frozen base)

In [ ]:
print("Building EfficientNetB0...")
model_b0, base_b0 = build_model(EfficientNetB0, IMG_SIZE_B0B1,
                                freeze_base=True, name='efficientnet_b0')
model_b0.compile(
    optimizer=keras.optimizers.Adam(P1_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model_b0.summary()


In [ ]:
print("=== Phase 1: Feature extraction (frozen base) ===")
t0 = time.time()

hist_b0_p1 = model_b0.fit(
    train_ds_b0,
    validation_data=val_ds_b0,
    epochs=P1_EPOCHS,
    class_weight=class_weight,
    callbacks=make_callbacks('b0_phase1'),
    verbose=1
)

print(f"Phase 1 done in {(time.time()-t0)/60:.1f} min")
print(f"Best val accuracy: {max(hist_b0_p1.history['val_accuracy']):.4f}")


## 10. EfficientNetB0 — Phase 2 (fine-tuning)

In [ ]:
print(f"Unfreezing last {P2_UNFREEZE_LAST} layers...")
base_b0.trainable = True

# Keep BatchNorm layers frozen to preserve learned statistics
for layer in base_b0.layers[:-P2_UNFREEZE_LAST]:
    layer.trainable = False
for layer in base_b0.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(1 for l in model_b0.layers if l.trainable)
print(f"Trainable layers: {trainable_count}")

model_b0.compile(
    optimizer=keras.optimizers.Adam(P2_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
print("=== Phase 2: Fine-tuning ===")
t0 = time.time()

hist_b0_p2 = model_b0.fit(
    train_ds_b0,
    validation_data=val_ds_b0,
    epochs=P2_EPOCHS,
    class_weight=class_weight,
    callbacks=make_callbacks('b0_phase2'),
    verbose=1
)

print(f"Phase 2 done in {(time.time()-t0)/60:.1f} min")
print(f"Best val accuracy: {max(hist_b0_p2.history['val_accuracy']):.4f}")

# Save final best model
model_b0.save(os.path.join(MODELS_DIR, 'best_model_b0.h5'))
print("Saved: models/best_model_b0.h5")


## 11. B0 training curves & evaluation

In [ ]:
plot_history(hist_b0_p1, hist_b0_p2, 'B0')
report_b0 = evaluate_model(model_b0, test_ds_b0, 'B0')


## 12. EfficientNetB1 comparison
> Skip this section if running on CPU — B1 is significantly slower.

In [ ]:
IMG_SIZE_B1 = (240, 240)
train_ds_b1 = make_dataset(TRAIN_DIR, IMG_SIZE_B1, shuffle=True,  augment=True)
val_ds_b1   = make_dataset(VAL_DIR,   IMG_SIZE_B1, shuffle=False, augment=False)
test_ds_b1  = make_dataset(TEST_DIR,  IMG_SIZE_B1, shuffle=False, augment=False)

model_b1, base_b1 = build_model(EfficientNetB1, IMG_SIZE_B1,
                                freeze_base=True, name='efficientnet_b1')
model_b1.compile(
    optimizer=keras.optimizers.Adam(P1_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("EfficientNetB1 built.")


In [ ]:
print("=== B1 Phase 1 ===")
t0 = time.time()
hist_b1_p1 = model_b1.fit(
    train_ds_b1, validation_data=val_ds_b1,
    epochs=P1_EPOCHS, class_weight=class_weight,
    callbacks=make_callbacks('b1_phase1'), verbose=1
)

print("=== B1 Phase 2 ===")
base_b1.trainable = True
for layer in base_b1.layers[:-P2_UNFREEZE_LAST]:
    layer.trainable = False
for layer in base_b1.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model_b1.compile(optimizer=keras.optimizers.Adam(P2_LR),
                 loss='categorical_crossentropy', metrics=['accuracy'])

hist_b1_p2 = model_b1.fit(
    train_ds_b1, validation_data=val_ds_b1,
    epochs=P2_EPOCHS, class_weight=class_weight,
    callbacks=make_callbacks('b1_phase2'), verbose=1
)
model_b1.save(os.path.join(MODELS_DIR, 'best_model_b1.h5'))
print(f"B1 done in {(time.time()-t0)/60:.1f} min")
plot_history(hist_b1_p1, hist_b1_p2, 'B1')
report_b1 = evaluate_model(model_b1, test_ds_b1, 'B1')


## 13. EfficientNetB3 comparison
> Skip on CPU. Very slow — recommend Colab GPU for B3.

In [ ]:
train_ds_b3 = make_dataset(TRAIN_DIR, IMG_SIZE_B3, shuffle=True,  augment=True)
val_ds_b3   = make_dataset(VAL_DIR,   IMG_SIZE_B3, shuffle=False, augment=False)
test_ds_b3  = make_dataset(TEST_DIR,  IMG_SIZE_B3, shuffle=False, augment=False)

model_b3, base_b3 = build_model(EfficientNetB3, IMG_SIZE_B3,
                                freeze_base=True, name='efficientnet_b3')
model_b3.compile(
    optimizer=keras.optimizers.Adam(P1_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("EfficientNetB3 built.")


In [ ]:
print("=== B3 Phase 1 ===")
t0 = time.time()
hist_b3_p1 = model_b3.fit(
    train_ds_b3, validation_data=val_ds_b3,
    epochs=P1_EPOCHS, class_weight=class_weight,
    callbacks=make_callbacks('b3_phase1'), verbose=1
)

print("=== B3 Phase 2 ===")
base_b3.trainable = True
for layer in base_b3.layers[:-P2_UNFREEZE_LAST]:
    layer.trainable = False
for layer in base_b3.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model_b3.compile(optimizer=keras.optimizers.Adam(P2_LR),
                 loss='categorical_crossentropy', metrics=['accuracy'])

hist_b3_p2 = model_b3.fit(
    train_ds_b3, validation_data=val_ds_b3,
    epochs=P2_EPOCHS, class_weight=class_weight,
    callbacks=make_callbacks('b3_phase2'), verbose=1
)
model_b3.save(os.path.join(MODELS_DIR, 'best_model_b3.h5'))
print(f"B3 done in {(time.time()-t0)/60:.1f} min")
plot_history(hist_b3_p1, hist_b3_p2, 'B3')
report_b3 = evaluate_model(model_b3, test_ds_b3, 'B3')


## 14. Model comparison summary

In [ ]:
def extract_f1(report_str):
    """Pull macro avg F1 from sklearn classification_report string."""
    for line in report_str.split('\n'):
        if 'macro avg' in line:
            return float(line.split()[4])
    return None

def best_val_acc(h1, h2):
    return max(h1.history['val_accuracy'] + h2.history['val_accuracy'])

results = {
    'Model':      ['EfficientNetB0', 'EfficientNetB1', 'EfficientNetB3'],
    'Params':     ['5.3M', '7.8M', '12M'],
    'Input size': ['224x224', '240x240', '300x300'],
    'Best val acc': [
        f"{best_val_acc(hist_b0_p1, hist_b0_p2):.4f}",
        f"{best_val_acc(hist_b1_p1, hist_b1_p2):.4f}",
        f"{best_val_acc(hist_b3_p1, hist_b3_p2):.4f}",
    ],
    'Macro F1': [
        f"{extract_f1(report_b0):.4f}",
        f"{extract_f1(report_b1):.4f}",
        f"{extract_f1(report_b3):.4f}",
    ],
}

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
df_results.to_csv(os.path.join(OUT_DIR, 'model_comparison.csv'), index=False)
print("\nSaved: outputs/model_comparison.csv")


## 15. Promote best model
Copy the best-performing variant to `models/best_model.h5` for use by gradcam.py and app.py.

In [ ]:
import shutil

# Change this to 'b1' or 'b3' if they scored higher
BEST_VARIANT = 'b0'

src  = os.path.join(MODELS_DIR, f'best_model_{BEST_VARIANT}.h5')
dest = os.path.join(MODELS_DIR, 'best_model.h5')

shutil.copy(src, dest)
print(f"Promoted {BEST_VARIANT} → models/best_model.h5")
print("Anuksha and Rishika can now load this file.")


## 16. Artifacts summary

In [ ]:
print("=== Outputs generated ===")
for f in sorted(os.listdir(OUT_DIR)):
    fpath = os.path.join(OUT_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  outputs/{f}  ({size_kb:.1f} KB)")

print()
print("=== Models saved ===")
for f in sorted(os.listdir(MODELS_DIR)):
    fpath = os.path.join(MODELS_DIR, f)
    size_mb = os.path.getsize(fpath) / (1024*1024)
    print(f"  models/{f}  ({size_mb:.1f} MB)")
